# Fine-tune ReactionT5 on Root-Aligned SMILES (Model 1, Kaggle GPU)

Tests the single most promising lever found in a literature review for pushing accuracy
further without a bigger model: **Root-Aligned SMILES** (Zhong et al., *Root-aligned SMILES:
a tight representation for chemical reaction prediction*, Chemical Science, 2022). Reported
effect: up to +9.23 top-1 points in one published ablation, from the data representation
alone -- no architecture change.

**What changed vs variant 2 (the proven 57k recipe this otherwise matches exactly):** only
the *target* (`reactants_smiles`) is rewritten so it starts ("is rooted") at the atom
corresponding to the product's own canonical-SMILES starting atom (found via RXNMapper
attention-guided atom mapping + RDKit `rootedAtAtom`) -- see `scripts/build_root_aligned_data.py`
for the exact method and docstring. **The product (input) is left untouched, still plain
canonical SMILES** -- this sidesteps the "what root do I pick at inference time, when I
don't have the reactants yet" problem entirely, since the input format never changes. A
checkpoint trained this way is used for inference exactly like any other; the model's raw
*output* is a valid (if non-canonical-rooted) SMILES that downstream canonicalization
(already used everywhere in this project's eval code) handles with no changes needed.

Applied to the exact same 57,000 ORD reactions as variant 2 (99.6% successfully root-aligned,
234 fell back to plain canonical -- RXNMapper mapping failures, kept rather than dropped so
pool size is unchanged), with the exact same hyperparameters as variant 2/the 150k-300k
"v2cfg" runs (`lr=5e-5`, 3 epochs, `--no-augment`, best-checkpoint-by-`eval_loss`) -- so this
experiment isolates *only* the representation change, not data volume or hyperparameters.

**Run this as Save & Run All (Commit), not an interactive Draft Session** -- same reasoning
as every other Kaggle notebook in this project (Draft Sessions aren't reliably persistent).

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and
**GPU accelerator** (T4x2). Kaggle's free GPU quota is **30 hours/week**.

**Data:** `kuzmenkooleh/retro-planner-ord-rootaligned-57k` (uploaded from
`data/v2_ord_train_rootaligned/{reactants_train,reactants_val}.jsonl`, built via
`scripts/build_root_aligned_data.py` from the same eval-excluded 57k pool as variant 2).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))
if torch.cuda.device_count() < 2:
    print("WARNING: fewer than 2 GPUs visible -- the torchrun --nproc_per_node=2 launch below expects 2.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

**Input data.** Adjust the dataset slug below to match whatever you named the Kaggle Dataset you uploaded (visible under `/kaggle/input/` once added as an input).

In [ ]:
import os, glob

dataset_slug = "retro-planner-ord-rootaligned-57k"  # @param {type:"string"}

candidates = [
    f"/kaggle/input/{dataset_slug}",
    f"/kaggle/input/datasets/kuzmenkooleh/{dataset_slug}",
]
base = next((c for c in candidates if os.path.exists(os.path.join(c, "reactants_train.jsonl"))), None)
if base is None:
    found = glob.glob("/kaggle/input/**/reactants_train.jsonl", recursive=True)
    listing = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "/kaggle/input MISSING"
    assert found, f"reactants_train.jsonl not found under /kaggle/input -- did you add the dataset as an input? /kaggle/input contents: {listing}"
    base = os.path.dirname(found[0])

train_file = os.path.join(base, "reactants_train.jsonl")
val_file = os.path.join(base, "reactants_val.jsonl")
assert os.path.exists(train_file), f"Not found: {train_file}"
assert os.path.exists(val_file), f"Not found: {val_file}"
print("Resolved base:", base)
print("Train file:", train_file, "--", sum(1 for _ in open(train_file)), "rows")
print("Val file:", val_file, "--", sum(1 for _ in open(val_file)), "rows")

**Cross-session resume on Kaggle.** There's no Drive-style live mount here -- `/kaggle/working` only persists once you **Save Version** ("commit") the notebook, which turns its contents into this notebook's own Output, downloadable as a dataset. To continue training in a later session:

1. This session: train, then **Save Version** before your quota/time runs out. The committed `/kaggle/working/<output_dir_name>` becomes an Output you can download or directly reuse.
2. Next session: either (a) add *this same notebook's* previous Output version as an input (Kaggle lets you pick a specific version's output), or (b) download the `final`/`checkpoint-N` folder and re-upload it as its own small Dataset -- same idea as the Colab notebook's cross-account resume.
3. Point `resume_from_checkpoint_path` below at wherever that folder landed under `/kaggle/input/...`.

Leave `resume_from_checkpoint_path` blank for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /kaggle/input/model1-rootaligned-checkpoint/checkpoint-NNNN  (full Trainer checkpoint)
# or   /kaggle/input/model1-rootaligned-checkpoint/final           (weights only)

In [ ]:
output_dir = "/kaggle/working/model1_reactant_rootaligned57k"  # @param {type:"string"}
time_budget_minutes = 150  # @param {type:"number"}
# 3 epochs (matching variant 2's proven config) over 57k/32 effective-batch is ~5,344 steps.
# Measured DDP rate from the 150k run's train.log: train_steps_per_second=0.972 -- so
# 5,344 / 0.972 = ~5,498s = ~92min (~1.5h). 150 min leaves ~60% headroom.

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate 5e-5 \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Same log-redirect reasoning as the Colab notebook: printing per-step output directly in the cell can make the tab unresponsive over a multi-hour run. Since this runs as a Commit job, you don't need to watch it at all -- check back later via `kaggle kernels status <user>/<slug>` (CLI) or the Output tab.

`--local-work-dir` points at `/kaggle/temp` (fast local scratch disk, wiped between sessions) so Trainer's own checkpoint rotation never touches `/kaggle/working` directly; the training script's own `DriveSyncCallback`-style logic still copies out one `latest_checkpoint` folder under `output_dir` after every save. Only rank 0 (of the 2 `torchrun` processes) does this Drive-style sync and the final save (fixed via `RANK`-based rank detection -- see `scripts/train_reactant_model_ord.py`), so there's no risk of the two GPU processes racing to write the same files.

**When done:** download via CLI (`kaggle kernels output <user>/<slug> -p <dest>`) or the Output tab. `output_dir/final` has the model. Evaluate it exactly like the other checkpoints:

```
python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model <downloaded_final_dir> \
    --num-beams 10 --output experiments/v2_model1_topk/ord150k_v2cfg_topk.json
```